# Fase 2 — Estandarización y anonimización de participaciones estudiantiles

**Trabajo de Grado:** Sistema de contraste de Redes Bayesianas y LSTM para la predicción explicable de la participación en el aula de los estudiantes de la Universidad Metropolitana
**Autores:** Nelson Carrillo, Luis Pérez — **Tutor:** Fernando Torre Mora — **Co-tutor:** José Alberto Peña E.

Este notebook implementa la **Fase 2 (Sprint 1)** de la metodología del anteproyecto: *"levantamiento, limpieza y estandarización de datos"*. Toma los registros crudos de participación en clase (archivos `.xlsx` con formatos heterogéneos según el docente y el trimestre) y produce, para cada uno, un archivo `.csv` estandarizado con una fila por estudiante y sesión/semana, siguiendo un esquema de columnas común.

## Qué hace este notebook

1. Extrae la tabla semana → tema de cada cronograma (`.docx` / `.xlsx`) de cada materia y trimestre.
2. Define funciones de limpieza (cédulas, artefactos en nombres, fechas en texto, códigos de asistencia P/1/T/F/J).
3. Define un extractor específico para cada uno de los **3 formatos de archivo fuente** detectados en la Fase 1 (por sesión de clase, agregado por semana, e híbrido con código de asistencia).
4. Ejecuta la extracción sobre los **11 archivos/hojas fuente** identificados como relevantes (los 10 originales + la hoja `ALGORITMOS` del libro de Estructuras de Datos 2526-2, que llena el hueco de Algoritmos 2526-2 — trimestre que de otra forma solo tenía cronograma, sin datos de participación).
5. Construye un **mapa de anonimización global** (cédula → `estudiante_id`), consistente entre todos los archivos.
6. Escribe los CSV estandarizados y anonimizados, y un **log de limpieza** documentando cada transformación.
7. Corre una **validación post-procesamiento** (cruce contra las columnas `TOTAL` de los archivos originales) para detectar errores de extracción.

## Cómo correrlo

Abre este notebook desde `preprocesamiento/` (con Jupyter, VS Code o similar) y ejecuta todas las celdas en orden (`Run All`). No requiere argumentos ni edición manual — las rutas son relativas a la ubicación del notebook dentro del repositorio.

**Salida** (se genera en `Datos Tesis/_procesado/`, que está en `.gitignore` salvo el propio código):
- `*_participaciones.csv` — 13 archivos estandarizados y anonimizados (ver Tabla 4 más abajo).
- `log_limpieza.txt` — bitácora pública de las transformaciones (sin datos personales).
- `_confidencial/mapeo_estudiantes.csv` — mapa cédula → `estudiante_id` (⚠️ contiene datos personales, **nunca se sube a git**, excluido vía `.gitignore`).
- `_confidencial/log_limpieza_detalle.txt` — bitácora con detalle que sí puede incluir nombres (p.ej. no-coincidencias en el cruce de la sección 6.3), también excluida de git.

## Esquema de salida (Tabla 4 del informe)

| Columna | Descripción |
|---|---|
| `estudiante_id` | Identificador anónimo consistente entre archivos (`anon_001`, `anon_002`, ...) |
| `numero_lista` | Número de lista en la sección, cuando el archivo fuente lo trae directamente |
| `materia` | Nombre de la asignatura |
| `trimestre` | Código del período académico |
| `seccion` | Identificador de sección, cuando está disponible en el archivo fuente |
| `fecha` | Fecha de la sesión (`YYYY-MM-DD`); vacía si el archivo fuente ya venía agregado por semana |
| `semana` | Número de semana dentro del trimestre |
| `tema` | Contenido académico de esa semana, tomado del cronograma correspondiente |
| `participaciones` | Cantidad de intervenciones del estudiante ese día/semana |
| `tipo_participacion` | Reservado para una futura clasificación cualitativa (no disponible en los datos actuales) |
| `asistencia` | Indicador de presencia (1/0), solo disponible en los 2 archivos que traen ese dato explícito |


## 0. Configuración y rutas

Todas las rutas son relativas a la carpeta de este notebook (`preprocesamiento/`), para que el repositorio sea portable entre máquinas.

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
import docx
from pathlib import Path
from datetime import datetime, timedelta

# Este notebook asume que se ejecuta con cwd = carpeta 'preprocesamiento/' del repo
# (comportamiento por defecto de Jupyter/VS Code al abrir un .ipynb).
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
BASE = REPO_ROOT / "Datos Tesis"
OUT_DIR = BASE / "_procesado"
CONF_DIR = OUT_DIR / "_confidencial"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CONF_DIR.mkdir(parents=True, exist_ok=True)

assert BASE.exists(), f"No se encontro la carpeta de datos crudos en {BASE}. Corre el notebook desde 'preprocesamiento/'."

LOG_LINES = []   # bitacora publica (sin datos personales) -> log_limpieza.txt
CONF_LINES = []  # bitacora confidencial (puede incluir nombres) -> _confidencial/log_limpieza_detalle.txt

def log(msg):
    LOG_LINES.append(msg)

def clog(msg):
    CONF_LINES.append(msg)

def flog(source, msg):
    log(f"[{source}] {msg}")

print("BASE:", BASE)
print("OUT_DIR:", OUT_DIR)


## 1. Cronogramas: extracción de la relación semana → tema

Los cronogramas se guardan en `.docx` (tabla `Tema | Semana | Actividad Evaluada | Fecha | Porcentaje`) para todas las materias, **excepto** Algoritmos y Programación 2425-2, cuyo cronograma es un `.xlsx` con un formato distinto (2 filas por semana: fecha y luego tema debajo de cada día de clase).

Se parsean ambos formatos a un diccionario común `{(materia, trimestre): {numero_semana: texto_tema}}`.

In [ ]:
def parse_docx_cronograma(path):
    """Extrae {semana:int -> tema:str} de un cronograma .docx con tabla Tema|Semana|..."""
    d = docx.Document(path)
    t = d.tables[0]
    result = {}
    for row in t.rows[1:]:
        cells_txt = [c.text.strip() for c in row.cells]
        if len(cells_txt) < 2:
            continue
        tema, semana = cells_txt[0], cells_txt[1]
        m = re.search(r'\d+', semana)
        if not m:
            continue
        wk = int(m.group())
        tema = re.sub(r'\s+', ' ', tema).strip()
        if not tema:
            continue
        result[wk] = tema if wk not in result else result[wk] + " | " + tema
    return result

def parse_xlsx_cronograma_alg2425_2(path, sheet):
    """Extrae {semana:int -> tema:str} del cronograma especial de Algoritmos 2425-2
    (2 filas por semana: fila de fechas, fila de temas por dia de clase)."""
    df = pd.read_excel(path, sheet_name=sheet, header=None)
    result = {}
    i = 7  # primera fila de datos 'Semana N'; fila 6 es el encabezado
    while i < len(df) - 1:
        semana_val = df.iat[i, 1]
        if pd.isna(semana_val):
            i += 1
            continue
        try:
            wk = int(semana_val)
        except (ValueError, TypeError):
            i += 1
            continue
        topic_row = df.iloc[i + 1]
        topics = []
        for col in (2, 3):
            v = topic_row[col]
            if isinstance(v, str) and v.strip():
                topics.append(re.sub(r'\s+', ' ', v.replace('\n', ' ')).strip())
        result[wk] = " | ".join(topics)
        i += 2
    return result

CRONOGRAMAS = {
    ("Algoritmos y Programación", "2425-2"): parse_xlsx_cronograma_alg2425_2(
        BASE / "Algoritmos y Programacion/2425-2/cronograma alg 2425-2.xlsx", "Cronograma LyM"),
    ("Algoritmos y Programación", "2526-1"): parse_docx_cronograma(
        BASE / "Algoritmos y Programacion/2526-1/Cronograma 2526-1.docx"),
    ("Algoritmos y Programación", "2526-2"): parse_docx_cronograma(
        BASE / "Algoritmos y Programacion/2526-2/Cronograma 2526-2.docx"),
    ("Algoritmos y Programación", "2526-3"): parse_docx_cronograma(
        BASE / "Algoritmos y Programacion/2526-3/Cronograma 2526-3.docx"),
    ("Computación Emergente", "2526-1"): parse_docx_cronograma(
        BASE / "Computacion Emergente/2526-1/Cronograma FPTSP25 2526-1.docx"),
    ("Computación Emergente", "2526-2"): parse_docx_cronograma(
        BASE / "Computacion Emergente/2526-2/Cronograma FPTSP25 2526-2 (3).docx"),
    ("Estructuras de Datos", "2425-3"): parse_docx_cronograma(
        BASE / "Estructura de Datos/2425-3/Cronograma Estructuras de Datos 2425-3.docx"),
    ("Estructuras de Datos", "2526-1"): parse_docx_cronograma(
        BASE / "Estructura de Datos/2526-1/Cronograma Estructuras de Datos 2526-1.docx"),
    ("Estructuras de Datos", "2526-2"): parse_docx_cronograma(
        BASE / "Estructura de Datos/2526-2/Cronograma Estructuras de Datos 2526-2.docx"),
    ("Estructuras de Datos", "2526-3"): parse_docx_cronograma(
        BASE / "Estructura de Datos/2526-3/Cronograma Estructuras de Datos 2526-3.docx"),
}

def tema_for(materia, trimestre, semana):
    d = CRONOGRAMAS.get((materia, trimestre), {})
    if semana is None:
        return ""
    try:
        return d.get(int(semana), "")
    except (ValueError, TypeError):
        return ""

for k, v in CRONOGRAMAS.items():
    print(f"{k}: {len(v)} semanas con tema")


## 2. Utilidades de limpieza

- `norm_cedula`: normaliza cédulas venidas como float/str a un string de dígitos.
- `clean_name_artifact`: repara el artefacto observado en varios archivos donde el nombre viene precedido por sus propias iniciales pegadas sin espacio (p. ej. `"FBFidel Eduardo Barreat Lemoine"` → `"Fidel Eduardo Barreat Lemoine"`).
- `norm_name_match`: normaliza un nombre completo (sin acentos, minúsculas) para poder cruzar por nombre cuando un archivo no trae cédula (caso Estructuras de Datos 2526-1).
- `monday_of` / `semana_from_date`: calculan el número de semana del trimestre a partir de la fecha de una sesión, cuando el archivo no trae ya una etiqueta de semana. **Método:** se asume que el lunes de la semana de la primera sesión registrada es el inicio de la "semana 1" del trimestre, y se cuentan bloques de 7 días desde ahí. Este método se validó contra el único cronograma que trae fechas explícitas por semana (Algoritmos 2425-2, ver sección 7) y reproduce exactamente la numeración real del profesor.
- `code_to_values`: traduce el código de asistencia (P/1/T/F/J) usado en los 2 archivos "híbridos" a `(participaciones, asistencia)`, documentando cualquier código no reconocido en vez de adivinar su significado.

In [ ]:
def norm_cedula(x):
    if pd.isna(x):
        return None
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        s = re.sub(r'\D', '', str(x))
        return s or None

DOUBLE_INITIAL_RE = re.compile(r'^[A-ZÁÉÍÓÚÑ]{2}([A-ZÁÉÍÓÚÑ][a-zà-ÿ].*)$')

def clean_name_artifact(s):
    if not isinstance(s, str):
        return s
    s = s.strip()
    m = DOUBLE_INITIAL_RE.match(s)
    return m.group(1) if m else s

def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def norm_name_match(s):
    s = clean_name_artifact(str(s))
    s = strip_accents(s).lower()
    s = re.sub(r'[^a-z ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def monday_of(d):
    return d - timedelta(days=d.weekday())

def semana_from_date(d, term_start):
    return (d.date() - term_start.date()).days // 7 + 1

def code_to_values(v):
    """Traduce un codigo de asistencia (P/1/T/F/J/numero) a (participaciones, asistencia, nota)."""
    if pd.isna(v):
        return 0.0, 0, None
    if isinstance(v, (int, float)):
        return float(v), 1, None
    s = str(v).strip()
    if s == 'P':
        return 0.0, 1, None
    if s == 'T':
        return 0.0, 1, None
    if s == 'F':
        return 0.0, 1, "codigo 'F' (Fraude) tratado como asistencia=1, participaciones=0"
    if s == 'J':
        return 0.0, 1, ("codigo 'J' (jubilado): estudiante asistio pero se retiro antes de terminar la clase "
                         "(confirmado por el autor de la tesis) -> asistencia=1, participaciones=0")
    if re.match(r'^\d+(\.\d+)?$', s):
        return float(s), 1, None
    return np.nan, np.nan, f"codigo especial no reconocido {s!r} tratado como NaN"

CEDULA_SET = set()
print("Utilidades cargadas OK")


## 3. Extractores por tipo de archivo fuente

La Fase 1 identificó 3 formatos distintos entre los 10 archivos relevantes:

1. **`process_simple_partic`** — una fila por estudiante, una columna por fecha de sesión de clase (6 archivos: Algoritmos 2425-2 sec1/sec2, Algoritmos 2526-1, Computación Emergente 2526-1, Estructuras 2425-3, y el insumo de Estructuras 2526-1).
2. **`process_hybrid_asistencia`** — columnas de sesión agrupadas bajo una etiqueta `Semana N` ya asignada por el docente, con código P/1/T/F/J en vez de solo un número, y columnas `prepa N` intercaladas que **no** son participación (son notas de preparaduría) y se ignoran (2 archivos: Algoritmos 2526-3, Estructuras 2526-3).
3. **`process_weekly_aggregated`** — ya viene agregado por semana (`Sem 1 ... Sem 12`), sin fecha de sesión individual; se mantiene esa granularidad tal cual, sin inventar fechas (2 archivos: Computación Emergente 2526-2, Estructuras 2526-2).

Un cuarto caso especial, **`process_estructura_2526_1`**, reutiliza el extractor 1 pero primero resuelve la cédula: la hoja `Hoja 1` de Estructuras de Datos 2526-1 no trae cédula, así que se cruza por nombre normalizado contra la hoja `Totales` del mismo libro (que sí la trae).

In [ ]:
def process_simple_partic(path, sheet, materia, trimestre, seccion, source_name,
                           header_row=0, text_date_map=None):
    df = pd.read_excel(path, sheet_name=sheet, header=None)
    header = df.iloc[header_row]
    data = df.iloc[header_row + 1:].reset_index(drop=True)

    date_cols = []
    for i, v in enumerate(header):
        if isinstance(v, (pd.Timestamp, datetime)):
            date_cols.append((i, pd.Timestamp(v).to_pydatetime()))
        elif text_date_map and isinstance(v, str) and v.strip() in text_date_map:
            date_cols.append((i, text_date_map[v.strip()]))

    cedula_idx = id_idx = None
    for i, v in enumerate(header):
        if isinstance(v, str) and v.strip().upper() in ('CEDULA', 'CÉDULA', 'CI'):
            cedula_idx = i
        if isinstance(v, str) and v.strip().upper() == 'ID':
            id_idx = i
    if cedula_idx is None:
        raise ValueError(f"No se encontro columna de cedula en {source_name}")
    if not date_cols:
        raise ValueError(f"No se encontraron columnas de fecha en {source_name}")

    all_dates = sorted(set(d for _, d in date_cols))
    term_start = monday_of(all_dates[0])

    rows = []
    n_students = 0
    special_codes = {}
    for _, r in data.iterrows():
        ced_raw = r[cedula_idx]
        if pd.isna(ced_raw):
            continue
        ced = norm_cedula(ced_raw)
        if ced is None:
            continue
        n_students += 1
        CEDULA_SET.add(ced)
        num_lista = r[id_idx] if id_idx is not None and not pd.isna(r[id_idx]) else None
        for ci, dt in date_cols:
            val = r[ci]
            if pd.isna(val):
                part_v = 0.0
            elif isinstance(val, (int, float)):
                part_v = float(val)
            else:
                part_v = np.nan
                special_codes[str(val)] = special_codes.get(str(val), 0) + 1
            wk = semana_from_date(dt, term_start)
            rows.append({
                'cedula_raw': ced, 'numero_lista': num_lista,
                'materia': materia, 'trimestre': trimestre, 'seccion': seccion,
                'fecha': dt.date().isoformat(), 'semana': wk,
                'tema': tema_for(materia, trimestre, wk),
                'participaciones': part_v, 'tipo_participacion': '', 'asistencia': np.nan,
            })

    flog(source_name, f"Sesiones detectadas: {len(date_cols)} (rango {all_dates[0].date()} a {all_dates[-1].date()}). "
                       f"Inicio de termino asumido (lunes semana 1): {term_start.date()}.")
    flog(source_name, f"Estudiantes procesados: {n_students}. Filas estudiante-fecha generadas: {len(rows)}.")
    if special_codes:
        flog(source_name, f"ADVERTENCIA: celdas con valores no numericos {special_codes} tratadas como NaN en 'participaciones'.")
    return rows

print("process_simple_partic definido")


In [ ]:
def process_hybrid_asistencia(path, sheet, materia, trimestre, seccion, source_name):
    df = pd.read_excel(path, sheet_name=sheet, header=None)
    row0 = df.iloc[0].tolist()
    row1 = df.iloc[1].tolist()
    data = df.iloc[2:].reset_index(drop=True)

    ff_row0, last = [], None
    for v in row0:
        if isinstance(v, str) and v.strip() and not v.strip().upper().startswith('P=PRESENTE'):
            last = v.strip()
        ff_row0.append(last)

    cedula_idx = id_idx = None
    date_cols = []
    for i, v in enumerate(row1):
        if isinstance(v, str) and v.strip().upper() in ('CEDULA', 'CÉDULA'):
            cedula_idx = i
        if isinstance(v, str) and v.strip().upper() == 'ID':
            id_idx = i
        if isinstance(v, (datetime, pd.Timestamp)):
            wk_label = ff_row0[i]
            m = re.search(r'\d+', wk_label) if wk_label else None
            date_cols.append((i, pd.Timestamp(v).to_pydatetime(), int(m.group()) if m else None))
    if cedula_idx is None:
        raise ValueError(f"No se encontro columna de cedula en {source_name}")

    rows = []
    n_students = 0
    code_notes = {}
    for _, r in data.iterrows():
        ced_raw = r[cedula_idx]
        if pd.isna(ced_raw):
            continue
        ced = norm_cedula(ced_raw)
        if ced is None:
            continue
        n_students += 1
        CEDULA_SET.add(ced)
        num_lista = r[id_idx] if id_idx is not None and not pd.isna(r[id_idx]) else None
        for ci, dt, wk in date_cols:
            part, asist, note = code_to_values(r[ci])
            if note:
                code_notes[note] = code_notes.get(note, 0) + 1
            rows.append({
                'cedula_raw': ced, 'numero_lista': num_lista,
                'materia': materia, 'trimestre': trimestre, 'seccion': seccion,
                'fecha': dt.date().isoformat(), 'semana': wk,
                'tema': tema_for(materia, trimestre, wk),
                'participaciones': part, 'tipo_participacion': '', 'asistencia': asist,
            })

    flog(source_name, f"Estudiantes procesados: {n_students}. Columnas de sesion usadas: {len(date_cols)} "
                       f"(se ignoraron columnas 'prepa N', que son notas de preparaduria, no participacion).")
    for note, cnt in code_notes.items():
        flog(source_name, f"{note} (x{cnt} celdas)")
    flog(source_name, f"Filas estudiante-fecha generadas: {len(rows)}.")
    return rows

print("process_hybrid_asistencia definido")


In [ ]:
def process_weekly_aggregated(path, sheet, materia, trimestre, seccion, source_name, header_row=1):
    df = pd.read_excel(path, sheet_name=sheet, header=None)
    header = df.iloc[header_row].tolist()
    data = df.iloc[header_row + 1:].reset_index(drop=True)

    cedula_idx = None
    sem_cols = []
    for i, v in enumerate(header):
        if isinstance(v, str) and v.strip().upper() in ('CEDULA', 'CÉDULA', 'CÉDULA DE IDENTIDAD'):
            cedula_idx = i
        if isinstance(v, str):
            m = re.match(r'^Sem\s*(\d+)$', v.strip(), re.IGNORECASE)
            if m:
                sem_cols.append((i, int(m.group(1))))
    if cedula_idx is None:
        raise ValueError(f"No se encontro columna de cedula en {source_name}")

    rows = []
    n_students = 0
    for _, r in data.iterrows():
        ced_raw = r[cedula_idx]
        if pd.isna(ced_raw):
            continue
        ced = norm_cedula(ced_raw)
        if ced is None:
            continue
        n_students += 1
        CEDULA_SET.add(ced)
        for ci, wk in sem_cols:
            val = r[ci]
            if pd.isna(val):
                part_v = 0.0
            elif isinstance(val, (int, float)):
                part_v = float(val)
            else:
                part_v = np.nan
            rows.append({
                'cedula_raw': ced, 'numero_lista': None,
                'materia': materia, 'trimestre': trimestre, 'seccion': seccion,
                'fecha': '', 'semana': wk,
                'tema': tema_for(materia, trimestre, wk),
                'participaciones': part_v, 'tipo_participacion': '', 'asistencia': np.nan,
            })

    wks = [w for _, w in sem_cols]
    flog(source_name, f"Estudiantes procesados: {n_students}. Semanas ya agregadas en el origen: Sem {min(wks)} a Sem {max(wks)} "
                       f"({len(sem_cols)} columnas). Sin fecha de sesion individual en el archivo fuente -> "
                       f"columna 'fecha' queda vacia, se mantiene la granularidad semanal original.")
    flog(source_name, f"Filas estudiante-semana generadas: {len(rows)}.")
    return rows

print("process_weekly_aggregated definido")


In [ ]:
def process_estructura_2526_1(path, materia, trimestre, seccion, source_name):
    """Caso especial: 'Hoja 1' no trae cedula -> se cruza por nombre normalizado contra 'Totales'."""
    h1 = pd.read_excel(path, sheet_name="Hoja 1", header=None)
    tot = pd.read_excel(path, sheet_name="Totales", header=None)

    header = h1.iloc[0].tolist()
    data = h1.iloc[1:].reset_index(drop=True)
    date_cols = [(i, pd.Timestamp(v).to_pydatetime()) for i, v in enumerate(header)
                 if isinstance(v, (datetime, pd.Timestamp))]
    all_dates = sorted(set(d for _, d in date_cols))
    term_start = monday_of(all_dates[0])

    tot_data = tot.iloc[1:].reset_index(drop=True)
    name_to_ced = {}
    for _, r in tot_data.iterrows():
        nom, ape, ced = r[0], r[1], r[2]
        if pd.isna(nom) or pd.isna(ape) or pd.isna(ced):
            continue
        name_to_ced[norm_name_match(f"{nom} {ape}")] = norm_cedula(ced)

    rows = []
    n_students = 0
    unmatched = []
    for _, r in data.iterrows():
        raw_name = r[0]
        if pd.isna(raw_name):
            continue
        clean = clean_name_artifact(str(raw_name).strip())
        ced = name_to_ced.get(norm_name_match(clean))
        if ced is None:
            unmatched.append(clean)
            continue
        n_students += 1
        CEDULA_SET.add(ced)
        for ci, dt in date_cols:
            val = r[ci]
            if pd.isna(val):
                part_v = 0.0
            elif isinstance(val, (int, float)):
                part_v = float(val)
            else:
                part_v = np.nan
            wk = semana_from_date(dt, term_start)
            rows.append({
                'cedula_raw': ced, 'numero_lista': None,
                'materia': materia, 'trimestre': trimestre, 'seccion': seccion,
                'fecha': dt.date().isoformat(), 'semana': wk,
                'tema': tema_for(materia, trimestre, wk),
                'participaciones': part_v, 'tipo_participacion': '', 'asistencia': np.nan,
            })

    flog(source_name, f"Sesiones detectadas: {len(date_cols)} (rango {all_dates[0].date()} a {all_dates[-1].date()}). "
                       f"Inicio de termino asumido: {term_start.date()}.")
    flog(source_name, "Cedula obtenida cruzando 'Hoja 1' (sin cedula) con 'Totales' (con cedula) por nombre normalizado.")
    flog(source_name, f"Estudiantes cruzados exitosamente: {n_students}. Sin match de cedula (excluidos del CSV): {len(unmatched)}.")
    if unmatched:
        clog(f"[{source_name}] Nombres de 'Hoja 1' sin match en 'Totales' (excluidos): {unmatched}")
    flog(source_name, f"Filas estudiante-fecha generadas: {len(rows)}.")
    return rows

print("process_estructura_2526_1 definido")


## 4. Ejecución sobre los 11 archivos/hojas fuente

Se listan explícitamente las 10 fuentes identificadas en la Fase 1, más la hoja `ALGORITMOS` del libro de Estructuras de Datos 2526-2 (agregada por decisión expresa del usuario para llenar el hueco de Algoritmos 2526-2, que no tenía archivo de participaciones propio). Algoritmos 2526-3 aporta 3 secciones (`sec 1`, `sec 3`, `sec 6`), por lo que se generan 3 CSV para esa fuente — **13 CSV en total**, nombrados `materia_trimestre[_seccion]_participaciones.csv`.

**Nota sobre duplicados:** Computación Emergente 2526-2 existe en dos libros (uno standalone y otro repetido dentro del libro de Estructuras de Datos 2526-2). Se usa el archivo *standalone*, tal como se acordó.

In [ ]:
SOURCES = {}

SOURCES['algoritmos_2425-2_sec1'] = process_simple_partic(
    BASE / "Algoritmos y Programacion/2425-2/Participacion sec 1 Alg 2425-2.xlsx", "Partic",
    "Algoritmos y Programación", "2425-2", "1", "algoritmos_2425-2_sec1",
    text_date_map={'Jan-06': datetime(2025, 1, 6), '08-Jan': datetime(2025, 1, 8), '13-Jan': datetime(2025, 1, 13)})

SOURCES['algoritmos_2425-2_sec2'] = process_simple_partic(
    BASE / "Algoritmos y Programacion/2425-2/Participacion sec 2 Alg 2425-2.xlsx", "Partic",
    "Algoritmos y Programación", "2425-2", "2", "algoritmos_2425-2_sec2")

SOURCES['algoritmos_2526-1'] = process_simple_partic(
    BASE / "Algoritmos y Programacion/2526-1/Participación Algoritmos 2526-1.xlsx", "Partic",
    "Algoritmos y Programación", "2526-1", "", "algoritmos_2526-1")
log("[algoritmos_2526-1] NOTA: el archivo fuente no distingue seccion por estudiante "
    "(el libro 'Total' menciona 'sec 1 y 5' de forma ambigua) -> columna 'seccion' se dejo vacia.")

_alg_2526_3 = BASE / "Algoritmos y Programacion/2526-3/Asistencia y participacion Algoritmos 2526-3.xlsx"
for sec in ['1', '3', '6']:
    SOURCES[f'algoritmos_2526-3_sec{sec}'] = process_hybrid_asistencia(
        _alg_2526_3, f"sec {sec}", "Algoritmos y Programación", "2526-3", sec, f"algoritmos_2526-3_sec{sec}")

SOURCES['computacion_emergente_2526-1'] = process_simple_partic(
    BASE / "Computacion Emergente/2526-1/Particip emergente 2526-1.xlsx", "Intervenciones",
    "Computación Emergente", "2526-1", "", "computacion_emergente_2526-1", header_row=1)
log("[computacion_emergente_2526-1] NOTA: seccion no especificada en el archivo fuente -> columna 'seccion' vacia.")

SOURCES['computacion_emergente_2526-2'] = process_weekly_aggregated(
    BASE / "Computacion Emergente/2526-2/Participaciones Emergente 2526-2.xlsx", "COMPURACION EMERGENTE",
    "Computación Emergente", "2526-2", "", "computacion_emergente_2526-2", header_row=1)
log("[computacion_emergente_2526-2] NOTA: se uso el archivo standalone de 'Computacion Emergente/2526-2/' "
    "(no la hoja duplicada dentro del libro de Estructuras de Datos 2526-2).")

SOURCES['estructuras_2425-3'] = process_simple_partic(
    BASE / "Estructura de Datos/2425-3/Participaciones Estructura de Datos 2425-3.xlsx", "Partic",
    "Estructuras de Datos", "2425-3", "", "estructuras_2425-3")
log("[estructuras_2425-3] NOTA: seccion no especificada en el archivo fuente -> columna 'seccion' vacia.")

SOURCES['estructuras_2526-1'] = process_estructura_2526_1(
    BASE / "Estructura de Datos/2526-1/Participaciones Estructura de Datos 2526-1.xlsx",
    "Estructuras de Datos", "2526-1", "", "estructuras_2526-1")

SOURCES['estructuras_2526-2'] = process_weekly_aggregated(
    BASE / "Estructura de Datos/2526-2/Participaciones Estructura de Datos 2526-2.xlsx", "ESTRUCTURAS-DE DATOS",
    "Estructuras de Datos", "2526-2", "", "estructuras_2526-2", header_row=1)
log("[estructuras_2526-2] NOTA: el libro fuente tambien contiene una hoja COMPURACION EMERGENTE para el mismo "
    "trimestre, que es duplicada del archivo standalone (ver nota de computacion_emergente_2526-2) y por eso "
    "no se proceso de nuevo aqui. La hoja ALGORITMOS del mismo libro SI se proceso, ver fuente 'algoritmos_2526-2' abajo.")

SOURCES['algoritmos_2526-2'] = process_weekly_aggregated(
    BASE / "Estructura de Datos/2526-2/Participaciones Estructura de Datos 2526-2.xlsx", "ALGORITMOS",
    "Algoritmos y Programación", "2526-2", "", "algoritmos_2526-2", header_row=1)
log("[algoritmos_2526-2] NOTA: Algoritmos 2526-2 no tenia archivo de participaciones propio (solo cronograma). "
    "Por decision expresa del usuario, se tomo esta hoja del libro de Estructuras de Datos 2526-2, que registra "
    "participaciones de Algoritmos para el mismo trimestre bajo el mismo formato Sem1..Sem12.")

SOURCES['estructuras_2526-3'] = process_hybrid_asistencia(
    BASE / "Estructura de Datos/2526-3/Asistencia y Participaciones Estructura de Datos 2526-3.xlsx", "Asistencia",
    "Estructuras de Datos", "2526-3", "2", "estructuras_2526-3")

print(f"\n{len(SOURCES)} fuentes procesadas, {sum(len(v) for v in SOURCES.values())} filas totales generadas.")


## 5. Anonimización global

Se recolectó una única cédula por estudiante a través de `CEDULA_SET` mientras se procesaba cada fuente. Ahora se asigna un `estudiante_id` (`anon_001`, `anon_002`, ...) por cédula, ordenando de forma ascendente para que la asignación sea **determinística y reproducible**: si vuelves a correr este notebook sobre los mismos datos, obtendrás exactamente el mismo mapeo.

Un mismo estudiante que participe en más de una materia/trimestre conserva el mismo `estudiante_id` en todos los CSV de salida.

⚠️ El mapeo cédula → `estudiante_id` se guarda en `Datos Tesis/_procesado/_confidencial/`, carpeta excluida de git — **nunca debe subirse al repositorio**.

In [ ]:
sorted_ceds = sorted(CEDULA_SET, key=lambda x: int(x))
anon_map = {c: f"anon_{i+1:03d}" for i, c in enumerate(sorted_ceds)}

log("")
log("=== Mapeo de anonimizacion ===")
log(f"Total de estudiantes unicos (por cedula) detectados en los 10 archivos: {len(sorted_ceds)}.")
log(f"IDs asignados: anon_001 a anon_{len(sorted_ceds):03d}, ordenados por cedula ascendente. "
    f"El mismo estudiante recibe el mismo estudiante_id en todos los CSV donde aparezca.")

mapping_path = CONF_DIR / "mapeo_estudiantes.csv"
pd.DataFrame({'cedula': sorted_ceds, 'estudiante_id': [anon_map[c] for c in sorted_ceds]}).to_csv(mapping_path, index=False)

print(f"{len(sorted_ceds)} estudiantes unicos anonimizados.")
print(f"Mapeo confidencial guardado en: {mapping_path}")


## 6. Escritura de los CSV estandarizados

Se aplica el mapa de anonimización, se ordenan las columnas según la Tabla 4, y se escribe un CSV por fuente.

In [ ]:
COLS = ['estudiante_id', 'numero_lista', 'materia', 'trimestre', 'seccion',
        'fecha', 'semana', 'tema', 'participaciones', 'tipo_participacion', 'asistencia']

written = []
for name, rows in SOURCES.items():
    for r in rows:
        r['estudiante_id'] = anon_map[r['cedula_raw']]
    df_out = pd.DataFrame(rows)
    if df_out.empty:
        log(f"[{name}] ADVERTENCIA: no se genero ninguna fila.")
        continue
    df_out = df_out[COLS]
    out_path = OUT_DIR / f"{name}_participaciones.csv"
    df_out.to_csv(out_path, index=False)
    written.append((name, out_path, len(df_out)))

log("")
log("=== Archivos CSV generados ===")
for name, path, n in written:
    log(f"{path.name}: {n} filas")

for name, path, n in written:
    print(f"{path.name}: {n} filas")


## 7. Validación post-procesamiento

Antes de dar por buena la extracción, se cruza automáticamente contra la propia columna `TOTAL`/`TOTAL` del archivo Excel original de Algoritmos 2425-2 sec 1 (el único de los 10 cuya hoja trae, además de las columnas de fecha, una columna de total ya calculada por el docente). Esta celda solo reporta **cifras agregadas** (no cédulas ni nombres), para que el notebook pueda compartirse sin exponer datos personales.

In [ ]:
_orig = pd.read_excel(BASE / "Algoritmos y Programacion/2425-2/Participacion sec 1 Alg 2425-2.xlsx", sheet_name="Partic", header=0)
_date_cols = list(_orig.columns[3:-1])
_sum_all = _orig[_date_cols].sum(axis=1, skipna=True)
_diff = (_sum_all - _orig['TOTAL']).abs()
_n_ok = int((_diff < 0.001).sum())
_n_total = len(_orig)

log("")
log("=== VALIDACION: suma de columnas de fecha vs columna TOTAL del Excel original (Algoritmos 2425-2 sec1) ===")
log(f"Coinciden exactamente: {_n_ok}/{_n_total} estudiantes.")
if _n_ok < _n_total:
    log(f"Los {_n_total - _n_ok} restantes tienen un TOTAL en el Excel que no coincide con la suma de las columnas de fecha; "
        f"en la revision manual (ver informe) se confirmo que se debe a que la formula TOTAL del archivo original "
        f"no fue extendida a la ultima columna de fecha (error del archivo fuente, no de esta extraccion).")

print(f"Coinciden exactamente: {_n_ok}/{_n_total} estudiantes.")
print("(ver detalle interpretativo en el log_limpieza.txt)")

# Semanas sin tema en el cronograma (informativo, no es error de cruce)
for _src in ['algoritmos_2526-1', 'computacion_emergente_2526-2']:
    _df = pd.DataFrame(SOURCES[_src])
    _huecos = sorted(_df.loc[_df['tema'] == '', 'semana'].unique().tolist())
    if _huecos:
        log(f"[{_src}] Semanas sin tema en el cronograma (no es error de cruce, el cronograma no registra tema esa semana): {_huecos}")
        print(f"[{_src}] semanas sin tema: {_huecos}")


## 8. Escritura de las bitácoras (`log_limpieza.txt`)

In [ ]:
with open(OUT_DIR / "log_limpieza.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(LOG_LINES))

with open(CONF_DIR / "log_limpieza_detalle.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(CONF_LINES) if CONF_LINES else "(sin observaciones confidenciales)")

print(f"Log publico escrito en: {OUT_DIR / 'log_limpieza.txt'} ({len(LOG_LINES)} lineas)")
print(f"Log confidencial escrito en: {CONF_DIR / 'log_limpieza_detalle.txt'} ({len(CONF_LINES)} lineas)")
print("\nProcesamiento completo.")
